In [22]:
import re
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from transformers import AutoTokenizer, AutoModel
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import CountVectorizer
from tqdm.auto import tqdm



# =========================================================
# 1. CONFIGURAÇÃO
# =========================================================

MODEL_NAME = "neuralmind/bert-base-portuguese-cased"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


# Labels-semente com descrições semânticas
LABEL_DESCRIPTIONS = {
    "ATO_PESSOAL": (
        "documento sobre nomeação, exoneração, designação, promoção, "
        "transferência, movimentação, aposentadoria, reforma ou pessoal militar"
    ),
    "CONTRATACAO": (
        "documento sobre licitação, contrato, extrato de contrato, fornecedor, "
        "serviço, aquisição, pregão, dispensa, inexigibilidade ou contratação administrativa"
    ),
    "ATO_NORMATIVO": (
        "documento sobre portaria, instrução, resolução, norma, diretriz, "
        "regulamento, delegação de competência ou ato normativo"
    ),
    "ORCAMENTARIO_FINANCEIRO": (
        "documento sobre orçamento, crédito, financeiro, empenho, dotação, "
        "pagamento, repasse, execução orçamentária ou recursos financeiros"
    ),
    "CONVENIO": (
        "documento sobre convênio, acordo, termo de cooperação, parceria institucional "
        "ou instrumento congênere"
    ),
    "DISCIPLINAR_SANCIONATORIO": (
        "documento sobre sindicância, processo administrativo, punição, penalidade, "
        "apuração, sanção ou responsabilização"
    )
}

LABEL_NAMES = list(LABEL_DESCRIPTIONS.keys())
LABEL_TEXTS = list(LABEL_DESCRIPTIONS.values())


# =========================================================
# 2. LIMPEZA E PREPARAÇÃO
# =========================================================

def normalize_whitespace(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = text.replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def remove_xml_tags(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = re.sub(r"<[^>]+>", " ", text)
    return normalize_whitespace(text)


def normalize_text(text: str) -> str:
    text = remove_xml_tags(text)
    text = text.replace("–", "-").replace("—", "-")
    text = normalize_whitespace(text)
    return text


def build_document_input(row: pd.Series, head_chars: int = 1200, tail_chars: int = 500) -> str:
    titulo = normalize_text(str(row.get("titulo", "") or ""))
    texto = normalize_text(str(row.get("texto", "") or ""))

    head = texto[:head_chars]
    tail = texto[-tail_chars:] if len(texto) > tail_chars else texto

    combined = f"TITULO: {titulo} [SEP] INICIO: {head} [SEP] FIM: {tail}"
    return combined.strip()


# =========================================================
# 3. HEURÍSTICAS POR REGEX
# =========================================================

HEURISTIC_PATTERNS = {
    "CONTRATACAO": [
        r"\bextrato de contrato\b",
        r"\blicita[cç][aã]o\b",
        r"\bpreg[aã]o\b",
        r"\bdispensa de licita[cç][aã]o\b",
        r"\binexigibilidade\b",
        r"\bfornecedor\b",
        r"\bobjeto:\b",
        r"\bcontrato n[ºo°]?\b",
    ],
    "ATO_NORMATIVO": [
        r"\bportaria\b",
        r"\binstru[cç][aã]o normativa\b",
        r"\bresolu[cç][aã]o\b",
        r"\bregulamento\b",
        r"\bdiretriz\b",
        r"\bdelega[cç][aã]o de compet[êe]ncia\b",
    ],
    "ATO_PESSOAL": [
        r"\bnomear\b",
        r"\bexonerar\b",
        r"\bdesignar\b",
        r"\bpromover\b",
        r"\btransferir\b",
        r"\bmovimenta[cç][aã]o\b",
        r"\baposentadoria\b",
        r"\breforma\b",
    ],
    "ORCAMENTARIO_FINANCEIRO": [
        r"\bempenho\b",
        r"\bdota[cç][aã]o\b",
        r"\bcr[eé]dito or[cç]ament[aá]rio\b",
        r"\bexecu[cç][aã]o or[cç]ament[aá]ria\b",
        r"\bpagamento\b",
        r"\brepasse\b",
    ],
    "CONVENIO": [
        r"\bconv[eê]nio\b",
        r"\btermo de coopera[cç][aã]o\b",
        r"\bacordo de coopera[cç][aã]o\b",
        r"\bparceria\b",
    ],
    "DISCIPLINAR_SANCIONATORIO": [
        r"\bsindic[âa]ncia\b",
        r"\bprocesso administrativo\b",
        r"\bpenalidade\b",
        r"\bsan[cç][aã]o\b",
        r"\bresponsabiliza[cç][aã]o\b",
        r"\bapura[cç][aã]o\b",
    ],
}


def heuristic_label(text: str):
    text_norm = normalize_text(text).lower()

    hits = []
    for label, patterns in HEURISTIC_PATTERNS.items():
        score = 0
        for pattern in patterns:
            if re.search(pattern, text_norm, flags=re.IGNORECASE):
                score += 1
        if score > 0:
            hits.append((label, score))

    if not hits:
        return None, 0

    hits = sorted(hits, key=lambda x: x[1], reverse=True)
    best_label, best_score = hits[0]

    # confiança heurística simples
    return best_label, best_score


# =========================================================
# 4. EMBEDDINGS COM BERTIMBAU
# =========================================================

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()


def mean_pooling(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    masked_embeddings = last_hidden_state * mask
    summed = masked_embeddings.sum(dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts


@torch.no_grad()
def encode_texts(
    texts,
    batch_size=16,
    max_length=256,
    show_progress=False,
    progress_desc="Gerando embeddings"
):
    all_embeddings = []

    batch_starts = range(0, len(texts), batch_size)
    if show_progress:
        total_batches = (len(texts) + batch_size - 1) // batch_size
        batch_starts = tqdm(batch_starts, total=total_batches, desc=progress_desc, leave=False)

    for i in batch_starts:
        batch = texts[i:i + batch_size]

        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        ).to(DEVICE)

        outputs = model(**inputs)
        embeddings = mean_pooling(outputs.last_hidden_state, inputs["attention_mask"])
        embeddings = F.normalize(embeddings, p=2, dim=1)
        all_embeddings.append(embeddings.cpu())

    return torch.cat(all_embeddings, dim=0)


LABEL_EMBS = encode_texts(LABEL_TEXTS)


# =========================================================
# 5. CLASSIFICAÇÃO SEMÂNTICA COM OUTROS
# =========================================================

def classify_with_outros(
    texts,
    threshold=0.55,
    margin=0.03,
    use_heuristics=True,
    show_progress=False
):
    """
    Regra:
    1. tenta heurística forte
    2. calcula similaridade com labels
    3. se score baixo ou ambíguo -> OUTROS
    """

    doc_embs = encode_texts(
        texts,
        show_progress=show_progress,
        progress_desc="Classificando documentos"
    )
    sims = torch.matmul(doc_embs, LABEL_EMBS.T)

    results = []

    text_indexes = range(len(texts))
    if show_progress:
        text_indexes = tqdm(text_indexes, total=len(texts), desc="Consolidando previsões", leave=False)

    for i in text_indexes:
        text = texts[i]
        heuristic_pred = None
        heuristic_score = 0

        if use_heuristics:
            heuristic_pred, heuristic_score = heuristic_label(text)

        scores = sims[i].tolist()
        ranked = sorted(
            zip(LABEL_NAMES, scores),
            key=lambda x: x[1],
            reverse=True
        )

        best_label, best_score = ranked[0]
        second_score = ranked[1][1] if len(ranked) > 1 else -1.0

        # Regra híbrida
        if heuristic_pred is not None and heuristic_score >= 2:
            final_label = heuristic_pred
            source = "heuristica"
        else:
            if best_score < threshold or (best_score - second_score) < margin:
                final_label = "OUTROS"
            else:
                final_label = best_label
            source = "similaridade"

        results.append({
            "texto_input": text,
            "label_predita": final_label,
            "melhor_label": best_label,
            "melhor_score": float(best_score),
            "segundo_score": float(second_score),
            "score_gap": float(best_score - second_score),
            "heuristica_label": heuristic_pred,
            "heuristica_score": heuristic_score,
            "fonte_predicao": source,
            "ranking_labels": ranked
        })

    return pd.DataFrame(results)


# =========================================================
# 6. APLICAR AO DATAFRAME
# =========================================================

def classify_dataframe(
    df: pd.DataFrame,
    title_col: str = "titulo",
    text_col: str = "texto",
    threshold: float = 0.55,
    margin: float = 0.03,
    use_heuristics: bool = True,
    show_progress: bool = False
) -> pd.DataFrame:
    df = df.copy()

    df[title_col] = df[title_col].fillna("")
    df[text_col] = df[text_col].fillna("")

    df["texto_modelo"] = df.apply(build_document_input, axis=1)

    pred_df = classify_with_outros(
        texts=df["texto_modelo"].tolist(),
        threshold=threshold,
        margin=margin,
        use_heuristics=use_heuristics,
        show_progress=show_progress
    )

    df = pd.concat([df.reset_index(drop=True), pred_df.reset_index(drop=True)], axis=1)
    return df


# =========================================================
# 7. CLUSTERIZAÇÃO DOS "OUTROS"
# =========================================================

def cluster_outros(
    df: pd.DataFrame,
    text_col: str = "texto_modelo",
    label_col: str = "label_predita",
    n_clusters: int = 8,
    top_terms: int = 12,
    random_state: int = 42,
    show_progress: bool = False
):
    """
    Clusteriza apenas documentos classificados como OUTROS.
    Retorna:
    - dataframe com cluster
    - resumo com palavras mais frequentes por cluster
    """

    df_outros = df[df[label_col] == "OUTROS"].copy()

    if df_outros.empty:
        print("Nenhum documento em OUTROS para clusterizar.")
        return df_outros, pd.DataFrame()

    # embeddings dos OUTROS
    outros_embs = encode_texts(
        df_outros[text_col].tolist(),
        show_progress=show_progress,
        progress_desc="Gerando embeddings de OUTROS"
    )
    X = outros_embs.numpy()

    # ajusta número de clusters ao volume disponível
    n_clusters = min(n_clusters, len(df_outros))
    if n_clusters < 2:
        df_outros["cluster_outros"] = 0
        return df_outros, pd.DataFrame({"cluster_outros": [0], "top_terms": ["amostra pequena"]})

    kmeans = KMeans(n_clusters=n_clusters, random_state=random_state, n_init=10)
    df_outros["cluster_outros"] = kmeans.fit_predict(X)

    # palavras mais frequentes por cluster
    vectorizer = CountVectorizer(
        stop_words=None,
        max_features=5000,
        ngram_range=(1, 2),
        lowercase=True
    )

    term_matrix = vectorizer.fit_transform(df_outros[text_col])
    vocab = np.array(vectorizer.get_feature_names_out())

    summaries = []

    for cluster_id in sorted(df_outros["cluster_outros"].unique()):
        mask = (df_outros["cluster_outros"] == cluster_id).to_numpy()
        cluster_term_sum = np.asarray(term_matrix[mask].sum(axis=0)).ravel()
        top_idx = cluster_term_sum.argsort()[::-1][:top_terms]
        terms = vocab[top_idx].tolist()

        summaries.append({
            "cluster_outros": int(cluster_id),
            "qtd_documentos": int(mask.sum()),
            "top_terms": terms
        })

    summary_df = pd.DataFrame(summaries).sort_values("qtd_documentos", ascending=False)
    return df_outros, summary_df


# =========================================================
# 8. FUNÇÃO PRINCIPAL DO PIPELINE
# =========================================================

def run_kdd_bootstrap_pipeline(
    df: pd.DataFrame,
    threshold: float = 0.55,
    margin: float = 0.03,
    use_heuristics: bool = True,
    n_clusters_outros: int = 8,
    show_progress: bool = True
):
    """
    1. classifica em labels conhecidas ou OUTROS
    2. clusteriza os OUTROS
    3. devolve tudo pronto para inspeção
    """

    pipeline_bar = tqdm(total=2, desc="KDD bootstrap", unit="etapa") if show_progress else None

    try:
        if pipeline_bar is not None:
            pipeline_bar.set_postfix_str("Classificação")

        df_classificado = classify_dataframe(
            df=df,
            threshold=threshold,
            margin=margin,
            use_heuristics=use_heuristics,
            show_progress=show_progress
        )

        if pipeline_bar is not None:
            pipeline_bar.update(1)
            pipeline_bar.set_postfix_str("Clusterização")

        df_outros, summary_clusters = cluster_outros(
            df_classificado,
            text_col="texto_modelo",
            label_col="label_predita",
            n_clusters=n_clusters_outros,
            show_progress=show_progress
        )

        if pipeline_bar is not None:
            pipeline_bar.update(1)
    finally:
        if pipeline_bar is not None:
            pipeline_bar.close()

    return df_classificado, df_outros, summary_clusters

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 12047.38it/s]
BertModel LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [23]:
import polars as pl

textos = (
    pl.read_parquet('../database/textos/*.parquet')
    .join(
        pl.read_parquet('../database/metadata/*.parquet'),
        on='id'
    )
    .with_columns(
        pl.col('pubDate').str.to_date(format='%d/%m/%Y')
    )
    .filter(
        pl.col('pubDate').dt.year()>=2018,
        pl.col('artCategory').str.to_lowercase().str.contains(r'(marinha|ex[ée]rcito|aeron[áa]utica)')
    )
).select('id', pl.col('name').alias('titulo'), 'texto').to_pandas()

In [ ]:
df_classificado, df_outros, summary_clusters = run_kdd_bootstrap_pipeline(
    textos,
    threshold=0.56,
    margin=0.04,
    use_heuristics=True,
    n_clusters_outros=4,
    show_progress=True
)

KDD bootstrap:   0%|          | 0/2 [00:00<?, ?etapa/s, Classificação]

In [ ]:
df_classificado

,id,titulo,texto,texto_modelo,texto_input,label_predita,melhor_label,melhor_score,segundo_score,score_gap,heuristica_label,heuristica_score,fonte_predicao,ranking_labels
0,515_20190102_11355821,Port_2082,"\nPORTARIA Nº 2.082, DE 27 DE DEZEMBRO DE 2018...",TITULO: Port_2082 [SEP] INICIO: PORTARIA Nº 2....,TITULO: Port_2082 [SEP] INICIO: PORTARIA Nº 2....,OUTROS,ATO_NORMATIVO,0.691899,0.660866,0.031033,ATO_NORMATIVO,1,similaridade,"[(ATO_NORMATIVO, 0.6918989419937134), (CONVENI..."
1,515_20190102_11355822,Port_2083,"\nPORTARIA Nº 2.083, DE 27 DE DEZEMBRO DE 2018...",TITULO: Port_2083 [SEP] INICIO: PORTARIA Nº 2....,TITULO: Port_2083 [SEP] INICIO: PORTARIA Nº 2....,OUTROS,ATO_NORMATIVO,0.686180,0.649267,0.036913,ATO_NORMATIVO,1,similaridade,"[(ATO_NORMATIVO, 0.6861799955368042), (CONVENI..."
2,515_20190104_11366759,Port-401-2018-DPC-ALTNOR-001-Mod,"\nPORTARIA Nº 401/DPC, DE 19 DE DEZEMBRO DE 20...",TITULO: Port-401-2018-DPC-ALTNOR-001-Mod [SEP]...,TITULO: Port-401-2018-DPC-ALTNOR-001-Mod [SEP]...,OUTROS,ATO_NORMATIVO,0.647746,0.610224,0.037522,ATO_NORMATIVO,1,similaridade,"[(ATO_NORMATIVO, 0.6477460265159607), (ATO_PES..."
3,515_20190107_11365668-1,20181210_Portaria_258 JJAER_RJJA,"\n\nPORTARIA DECEA Nº 258/JJAER, 10 DE DEZEMBR...",TITULO: 20181210_Portaria_258 JJAER_RJJA [SEP]...,TITULO: 20181210_Portaria_258 JJAER_RJJA [SEP]...,ATO_NORMATIVO,ATO_NORMATIVO,0.698710,0.651132,0.047578,ATO_NORMATIVO,2,heuristica,"[(ATO_NORMATIVO, 0.6987103819847107), (CONTRAT..."
4,515_20190107_11365668-2,20181210_Portaria_258 JJAER_RJJA,"\nArt. 38 As sessões serão públicas, ressalvad...",TITULO: 20181210_Portaria_258 JJAER_RJJA [SEP]...,TITULO: 20181210_Portaria_258 JJAER_RJJA [SEP]...,OUTROS,ATO_NORMATIVO,0.675199,0.671516,0.003684,NaN,0,similaridade,"[(ATO_NORMATIVO, 0.6751994490623474), (CONTRAT..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,515_20200122_12369101,Port-12-2020-DPC-DESP-651---STAR,"PORTARIA Nº 12/DPC, DE 21 DE JANEIRO DE 2020\n...",TITULO: Port-12-2020-DPC-DESP-651---STAR [SEP]...,TITULO: Port-12-2020-DPC-DESP-651---STAR [SEP]...,OUTROS,ATO_NORMATIVO,0.642832,0.626637,0.016194,ATO_NORMATIVO,1,similaridade,"[(ATO_NORMATIVO, 0.6428315043449402), (CONTRAT..."
96,515_20200122_12369439,Port-13-2020-DPC-DESP-651---CBO-,"PORTARIA Nº 13/DPC, DE 21 DE JANEIRO DE 2020\n...",TITULO: Port-13-2020-DPC-DESP-651---CBO- [SEP]...,TITULO: Port-13-2020-DPC-DESP-651---CBO- [SEP]...,OUTROS,ATO_NORMATIVO,0.652056,0.630568,0.021487,ATO_NORMATIVO,1,similaridade,"[(ATO_NORMATIVO, 0.6520555019378662), (CONTRAT..."
97,515_20200123_12373392,Port-14-2020-DPC-DESP-651---STAR,"PORTARIA Nº 14/DPC, DE 22 DE JANEIRO DE 2020\n...",TITULO: Port-14-2020-DPC-DESP-651---STAR [SEP]...,TITULO: Port-14-2020-DPC-DESP-651---STAR [SEP]...,OUTROS,ATO_NORMATIVO,0.645565,0.628348,0.017217,ATO_NORMATIVO,1,similaridade,"[(ATO_NORMATIVO, 0.6455645561218262), (CONTRAT..."
98,515_20200123_12373581,Extrato portaria CPCE 01 2020,"\nPORTARIA Nº 1/CPCE, DE 9 DE JANEIRO DE 2020\...",TITULO: Extrato portaria CPCE 01 2020 [SEP] IN...,TITULO: Extrato portaria CPCE 01 2020 [SEP] IN...,ATO_NORMATIVO,ATO_NORMATIVO,0.703767,0.665959,0.037809,ATO_NORMATIVO,2,heuristica,"[(ATO_NORMATIVO, 0.7037673592567444), (CONTRAT..."


In [ ]:
df_classificado.to_csv('df_classificado.csv', index=False)
df_outros.to_csv('df_outros.csv', index=False)